# barbi_exp1: adjustment of a warm anomaly in a closed basin on the sphere
pyOM2: for_config/barbi_exp1.f90

A Gaussian buoyancy anomaly in a uniformly stratified closed basin radiates
Rossby waves. Buoyancy is the prognostic tracer (eq_of_state=0).

In [ ]:
import sys; sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.pe_realistic.model import model

In [ ]:
nx, ny, nz = 20, 20, 15
dx, dt     = 2.0, 3600.
dzt        = np.array([50., 70., 100., 140., 190., 240., 290., 340., 390., 440., 490., 540., 590., 640., 690.])[::-1]
N_0        = 2.6e-3
x0, y0, x_len = 30.0, 48.0, 8.0      # position and width of the anomaly in degrees
b_anomaly  = 0.5*9.81*1.67e-4         # 0.5 K
omega      = 7.2921e-5
nsteps     = 8760                     # 1 year
plot_every = 240

par = dict(nx=nx, ny=ny, nz=nz, dxt=dx, dyt=dx, dzt=dzt, dt=dt,
           coord_degree=True, x_origin=0.0, y_origin=30.0, set_zero_at_surface=True,
           periodic_in_x=False, periodic_in_y=False,
           eq_of_state=0, implicit_free_surface=False, explicit_free_surface=False,
           epsilon=1e-12, max_iterations=50000, superbee_advection=True,
           Ah=2e3, vmix_scheme='explicit', kappa_T=1e-4, A_v=1e-3)

In [ ]:
class barbi_exp1(model):

    def set_coriolis(self):
        f = 2*omega*np.sin(self.yt/180.*np.pi)
        self.coriolis_t = OM.np.asarray(f[:, None] + 0*self.xt[None, :])

    def set_initial_conditions(self):
        Z, Y, X = np.meshgrid(self.zt_gl, self.yt_gl, self.xt_gl, indexing='ij')
        b = N_0**2*Z + b_anomaly*np.exp(-(X - x0)**2/x_len**2 - (Y - y0)**2/x_len**2)*np.exp(Z/500.)
        zeros = np.zeros((nz, ny, nx))
        super().set_initial_conditions(zeros, zeros, b, zeros, np.zeros((ny, nx)))

In [ ]:
M = barbi_exp1(par)
M.set_coriolis()
M.set_topography()
M.prepare_for_run()
M.set_initial_conditions()
h = M.halo
ks = -h - 1                         # surface level
b0 = M.temp[ks, h:-h, h:-h]*1        # initial surface buoyancy

In [ ]:
def plot(t, tau):
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    c = ax[0].pcolormesh(M.xt_gl, M.yt_gl, M.temp[ks, h:-h, h:-h] - b0, cmap='RdBu_r')
    plt.colorbar(c, ax=ax[0], label='surface b anomaly [m/s$^2$]')
    c = ax[1].pcolormesh(M.xt_gl, M.yt_gl, M.p_s[tau, h:-h, h:-h], cmap='RdBu_r')
    plt.colorbar(c, ax=ax[1], label='$p_s$ [m$^2$/s$^2$]')
    for a in ax: a.set_xlabel('lon'); a.set_ylabel('lat')
    fig.suptitle('t = %.0f days' % (t/86400)); plt.tight_layout(); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step_implicit_pressure(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt, tau)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
plot(nsteps*dt, taum1)